# QIHGNN-MSCA: training on Kaggle

Free GPU training for the full model, the matched baseline and the three ablations.

**Before running**
1. *Add data* (right panel) → search and add these three datasets:
   `awsaf49/brats2020-training-data`, `harshwardhanbhangale/lits-dataset`, `balajikartheek/cervical-cancer-segmentation`.
2. *Settings* → Accelerator **GPU T4 x2** (or P100), Internet **on**.
3. Put your GitHub repository URL in the first code cell.

A Kaggle session lasts up to 12 hours. Checkpoints are written every epoch; if a session ends, start a new one and use the resume cell.

In [ ]:
REPO = "https://github.com/<your-username>/qihgnn-msca.git"   # <- change this
!git clone -q $REPO /kaggle/working/qihgnn-msca
%cd /kaggle/working/qihgnn-msca

In [ ]:
# torch is preinstalled on Kaggle; install the rest
!pip install -q nibabel h5py starlette pydantic
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 1. Link the Kaggle inputs into `data/raw/`

In [ ]:
import glob, os
links = {"brats2020": "brats2020-training-data", "lits": "lits-dataset", "cervical": "cervical-cancer-segmentation"}
os.makedirs("data/raw", exist_ok=True)
for key, name in links.items():
    found = glob.glob(f"/kaggle/input/{name}") or glob.glob(f"/kaggle/input/*{name.split('-')[0]}*")
    if found and not os.path.exists(f"data/raw/{key}"):
        os.symlink(found[0], f"data/raw/{key}")
    print(key, "->", found[0] if found else "NOT ATTACHED")

## 2. Inspect, decide supervision, preprocess once

In [ ]:
%cd ml
!python -m qihgnn.data.inspection
!python -m qihgnn.data.cache
!python -m qihgnn.verify

## 3. Train
Start with `dev.json` (about an hour) to check the loss falls, then run `full.json`.

In [ ]:
!python -m qihgnn.train --config configs/dev.json

In [ ]:
!python -m qihgnn.train --config configs/full.json

In [ ]:
# Resume after a session ends (adjust the run folder):
# !python -m qihgnn.train --config configs/full.json --resume ../runs/<run-id>/last.pt

## 4. Baseline and ablations (same schedule)

In [ ]:
for cfg in ["baseline", "ablate_mpo", "ablate_hypergraph", "ablate_msca"]:
    !python -m qihgnn.train --config configs/{cfg}.json

## 5. Evaluate and export for the website

In [ ]:
import glob
runs = sorted(glob.glob("../runs/*/best.pt"))
print("\n".join(runs))
# Order the checkpoints to match the labels below before running:
# !python -m qihgnn.evaluate --checkpoint <full> <baseline> <no-mpo> <no-hypergraph> <no-msca> --label full baseline no-mpo no-hypergraph no-msca

In [ ]:
!python -m qihgnn.export_web --cases-from-cache 3
!cd .. && zip -qr /kaggle/working/qihgnn-results.zip results web/public/data runs/*/config.json runs/*/history.csv
print("Download /kaggle/working/qihgnn-results.zip from the Output tab and unzip it over your local repository.")